In [2]:
import sys
from pathlib import Path

from matplotlib.animation import FuncAnimation
from pathlib import Path
python_dir = Path.cwd().resolve().parents[0]
sys.path.insert(0, str(python_dir))

import integration.rhs as rhs
import numpy as np
import matplotlib.pyplot as plt
import h5py
import scipy.special as sp
import FourierSeries as fs
from scipy.integrate import solve_ivp
import os
from matplotlib.widgets import Slider
from scipy.linalg import null_space
from scipy.interpolate import interp1d

In [2]:
%matplotlib Qt

In [ ]:
## base energy scale:
alpha_hammaker = 3.5e-24 # 6.3 https://arxiv.org/html/2504.13001v1#S5

L = 1e-3 # 1 mm
L0 = L / (2*np.pi)
h = 15e-9
rho = 150 # kg/m^3
E0 = 3.0*rho*alpha_hammaker/h**4*L0**4
g = 3*alpha_hammaker / h**4

_t0 = np.sqrt(L0 / g)
hbar = 	1.054571817e-34 # J*s

hbar_0 = hbar / (E0 * _t0)
G = 20 # Mhz / nm

G = G * 1e6 * 1e9 # Hz / m # factor of 2pi might be missing if the original equation is hbar G_w with G_w in rad/s / m

G0 = G * _t0 * L0

print(f"Base energy scale: {E0} J = 1 a.u. of Energy")
print(f"Base time scale: {_t0} s = 1 a.u. of Time")
print(f"Base length scale: {L0} m = 1 a.u. of Length")

print(f"value of hbar G in rescaled units: {hbar_0 * G0}")

Base energy scale: 1.996163216188622e-05 J = 1 a.u. of Energy
Base time scale: 8.759875512285277e-07 s = 1 a.u. of Time
Base length scale: 0.00015915494309189535 m = 1 a.u. of Length
value of hbar G in rescaled units: 1.681629199053351e-17


In [9]:
R1 = 50e3
def v_error(Rf, R1, Vos, Ib):
    return (Rf/R1 +1) * (Vos + Ib * (1/(1/R1 + 1/Rf)))

Rf = np.linspace(1e6, 100e6, 100)
Vos = 0.35e-6
Ib = 30e-12 # A
Verror = v_error(Rf, R1, Vos, Ib)

%matplotlib Qt
plt.figure()
plt.plot(Rf/1e6, Verror * 1e6) 
plt.grid()
plt.xlabel("Rf (MOhms)")
plt.ylabel("Voltage error (uV)")
plt.title("Voltage error as a function of feedback resistor Rf")


Text(0.5, 1.0, 'Voltage error as a function of feedback resistor Rf')

In [ ]:

simManager = rhs.SimulationManager(r"D:\repos\superfluid-dynamics\CuSuperHelium\x64\Release\CuSuperHelium.dll")


N = 256
t0 = 0.0
t1 = 1500 # in us #~ 1 au of time is about 1 us in for this system (L ~ 1 mm, depth 20 nm)

sim_props = rhs.CSimulationProperties(
        L = 1e-3,
        depth = 15e-9,
        rho = 150,
        kappa = 0,
        use_expansions = False,
        infinite_depth = False
    )
L0 = sim_props.L / (2.0 * np.pi)
optomechanical_props = rhs.COptomechanicalProperties(
    detuning = 0.1,
    gamma = 0.01,
    G = -1e-6,
    tau = 1.0,
    max_intensity = 0.01,
    initial_time = 0.0,
    location_x0_mode = np.pi * 1.0,
    sigma_optical_mode =  20e-6 / L0,
    beta =  1.0,
    damping_strength = 0.0
)

rk4_props = rhs.CRK4Options(
    timeStep = 0.01,
    t0 = t0,
    t1 = t1,
    returnTrajectory = True,
)



r = np.array([2.0*np.pi/N*x for x in range(N)])


def gaussian(x, x0 = 0.75*np.pi, sigma=0.8):
    return np.exp(-(x - x0)**2 / (2 * sigma**2))

def soliton_sech2(x, x0, sigma):
    return 1 / np.cosh((x - x0) / sigma)**2

def setInitialY0(r):
    pot = np.zeros_like(r)
    ampl = np.zeros_like(r)#np.ones_like(r) * 0.01 * sim_props.depth / L0 * soliton_sech2(r, x0=np.pi, sigma=0.8)
    pot = np.zeros_like(r) # np.ones_like(r) *  0.01 * soliton_sech2(r, x0=np.pi, sigma=0.8)
    delayed = np.zeros_like(r)
    return np.concatenate((r, ampl, pot, delayed))

Y0 = setInitialY0(r)
g = 3*2.6e-24 / sim_props.depth**4
_t0 = np.sqrt(L0 / g)

print(f"Reference Time is {_t0:.2e} s")
print(f"Reference Length is {L0:.2e} m")
# np.exp(-rk4_props.timeStep / 0.01)
print(f"Depth in non-dim is {sim_props.depth / L0:.7e}")

def f(y):
    return simManager.calculate_augmented_rhs(y, sim_props, optomechanical_props)


dampings = np.array([0.0, 0.1, 0.2, 2])*-1e-2
def interpolate(x, y, x0):
    f = interp1d(x, y, fill_value="extrapolate")
    return f(x0)

def plot_phase_diagram(value, change_func, ax_pd, ax_ts, label="Detuning", n_arrows=5, lw=0.8, arrow_step=5):
    change_func(value)
    # t1 t0 to non diamensional time
    L0 = sim_props.L / (2.0 * np.pi)
    g = 3*2.6e-24 / sim_props.depth**4
    _t0 = np.sqrt(L0 / g)

    rk4_props.t0 =  1e-6 * rk4_props.t0 / _t0 # t0, t1 timeStep are in us, convert to non-dimensional time
    rk4_props.t1 = 1e-6 * rk4_props.t1 / _t0
    rk4_props.timeStep = 1e-6 * rk4_props.timeStep / _t0
    print(f"Integrating for {label}={value:.3e} with time step {rk4_props.timeStep:.2e} and total time {rk4_props.t1:.2e} a.u.t")
    res, T_new, Y_new = simManager.integrate_augmented_optomechanical_problem(Y0, sim_props, optomechanical_props, rk4_props)
    if res != 0:
        print("Error in integration")
        exit(1)
    x0 = 1.0*np.pi

    

    values_y = np.zeros_like(T_new)
    values_phi = np.zeros_like(T_new)
    values_d = np.zeros_like(T_new)
    # plt.figure()
    for i, t in enumerate(T_new):
        values_y[i] = L0 * interpolate(Y_new[i, :N], Y_new[i, N:2*N], x0)
        values_phi[i] = interpolate(Y_new[i, :N], Y_new[i, 2*N:3*N], x0)
        values_d[i] = interpolate(Y_new[i, :N], Y_new[i, 3*N:4*N], x0)

    ydot = np.gradient(values_y, T_new * _t0)
    line, = ax_pd.plot(values_y, ydot, label=f"{label}={value:.3e}", lw=0.8)
    color = line.get_color()
    idx = np.linspace(0, len(values_y) - arrow_step - 1, n_arrows, dtype=int)
    for i in idx:
        ax_pd.annotate(
            "",
            xy=(values_y[i + arrow_step], ydot[i + arrow_step]),
            xytext=(values_y[i], ydot[i]),
            arrowprops=dict(
                arrowstyle="->",
                color=color,
                lw=lw,
                shrinkA=0,
                shrinkB=0,
            ),
        )
    
    ax_ts.plot(T_new*_t0 *1e6, values_y, label=f"{label}={value:.3e}", lw=lw)

    return T_new, Y_new, L0, _t0

# xdot = np.gradient(values, T_new)
## make a subplot with the 3 phase diagrams for (y, ydot), (phi, phi_dot), (d, d_dot)
fig, axes = plt.subplots(2, 1, figsize=(15, 5))

change_detuning = lambda detuning: setattr(optomechanical_props, "detuning", detuning)
change_optical_sigma = lambda sigma: setattr(optomechanical_props, "sigma_optical_mode", sigma)
change_gamma = lambda gamma: setattr(optomechanical_props, "gamma", gamma)
change_damping = lambda damping: setattr(optomechanical_props, "damping_strength", damping)

change_depth = lambda depth: setattr(sim_props, "depth", depth)

depths = np.array([ 15, 20 ])*1e-9

times = []
interfaces = []
t0s = []
L0s = []
for depth in depths:
    T, Y, L0, t0 = plot_phase_diagram(depth, change_depth, axes[0], axes[1], label="Depth")
    times.append(T)
    interfaces.append(Y)
    t0s.append(t0)
    L0s.append(L0)

Reference Time is 1.02e-06 s
Reference Length is 1.59e-04 m
Depth in non-dim is 9.4247780e-05
Integrating for Depth=1.500e-08 with time step 9.84e-03 and total time 1.48e+03 a.u.t


In [ ]:
axes[0].set_xlabel(r"$y(x_0 = \pi)$ m")
axes[0].set_ylabel(r"$\dot{y}(x_0 = \pi)$ m/s")
axes[0].set_title("Phase Diagram for y at x0 = pi")
axes[0].legend()

axes[1].set_xlabel("Time (µs)")
axes[1].set_ylabel(r"$y(x_0 = \pi)$ m")
axes[1].set_title("Time Series for y at x0 = pi")
axes[1].legend()

In [ ]:
plt.figure()
plt.plot(interfaces[0][-1, :N], 1e9 * L0s[0] * interfaces[0][-1, N:2*N], label=f"Depth={depths[0]*1e9:.0f} nm")
plt.plot(interfaces[1][-1, :N], 1e9 * L0s[1] * interfaces[1][-1, N:2*N], label=f"Depth={depths[1]*1e9:.0f} nm")

In [ ]:
%matplotlib Qt

In [ ]:
(times[1][-1] * t0s[1])*1e6

295.1627142790162

In [ ]:
frames

29999

In [ ]:

# axes[1].scatter(values_phi, phi_dot)
# axes[1].set_xlabel(r"$\phi(x_0 = \pi)$")
# axes[1].set_ylabel(r"$\dot{\phi}(x_0 = \pi)$")
# axes[1].set_title("Phase Diagram for phi at x0 = pi")
# axes[2].scatter(values_d, d_dot)
# axes[2].set_xlabel(r"$d(x_0 = \pi)$")
# axes[2].set_ylabel(r"$\dot{d}(x_0 = \pi)$")
# axes[2].set_title("Phase Diagram for d at x0 = pi")
interpolations = []
for i, data in enumerate(times):
    L0, t0 = L0s[i], t0s[i]
    interp_func_x, interp_func_y = interp1d(times[i]*t0, interfaces[i][:, :N], axis=0), interp1d(times[i]*t0, interfaces[i][:, N:2*N], axis=0)
    interpolations.append((interp_func_x, interp_func_y))

tmin, tmax = min(times[0][0]*t0s[0], times[1][0]*t0s[1]), max(times[0][-1]*t0s[0], times[1][-1]*t0s[1])
frames = max(len(times[0]), len(times[1]))
ts = np.linspace(tmin, tmax, frames)

#



print(f"{tmin:.4e} s, {tmax:.4e} s, frames: {frames}")
print(f"{times[0][0]*t0s[0]} s")
print(f"{times[1][0]*t0s[1]} s")



    
    # ax1.set_ylim(interfaces[0][:, N:2*N].min()* L0s[0] * 1e9, interfaces[0][:, N:2*N].max() * L0s[0] * 1e9)





9.8391e-09 s, 2.9999e-04 s, frames: 29999
1e-08 s
9.83908511214087e-09 s


In [ ]:
len(times)

2

In [ ]:
values_x = [[], []]
values_y = [[], []]
for i, data in enumerate(times):
    for t in ts:
        t0, L0 = t0s[i], L0s[i]
        if t <= times[i][0]*t0 or t >= times[i][-1]*t0:
            values_x[i].append(None)
            values_y[i].append(None)
            continue
        x = interpolations[i][0](t)*L0 * 1e3
        y = interpolations[i][1](t)*L0 * 1e9
        values_x[i].append(x)
        values_y[i].append(y)

In [ ]:
## figure for the surface of the superfluid:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4), sharey=True, sharex=True)

time_text = fig.suptitle(f"t = {times[0][0]:.4f}")

line1, = ax1.plot(interfaces[0][0, :N]*1e3, interfaces[0][0, N:2*N]* L0s[0] *1e9, lw=0.8)
line2, = ax2.plot(interfaces[1][0, :N]*1e3, interfaces[1][0, N:2*N]* L0s[1] * 1e9, lw=0.8)

data_axes = [(ax1, line1, interfaces[0], depths[0], L0s[0], t0s[0]), (ax2, line2, interfaces[1], depths[1], L0s[1], t0s[1])]

def init():
    for ax, line, Y, depth,L0, t0 in data_axes:
        ax.set_title(f"Depth = {depth*1e9:.0f} nm")
        ax.set_xlim(Y[0, 0]*L0*1e3, Y[0, N-1]*L0*1e3)
        ax.set_ylabel("y (nm)")
        ax.set_xlabel("x (mm)")
    ax1.set_ylim(interfaces[0][:, N:2*N].min()* L0s[0] * 1e9, interfaces[0][:, N:2*N].max() * L0s[0] * 1e9)
    ax2.set_ylim(interfaces[0][:, N:2*N].min()* L0s[0] * 1e9, interfaces[0][:, N:2*N].max() * L0s[0] * 1e9)

def update(frame):
    t = tmin + frame * (tmax - tmin) / frames
    time_text.set_text(f"t = {t*1e6:.4f} us")
    for i, data in enumerate(data_axes):
        

        ax, line, Y, depth, L0, t0 = data

        x, y = values_x[i][frame], values_y[i][frame]
        if x is None or y is None:
            # ax.clear()
            ax.set_title(f"no data for t = {t*1e6:.4f} us")
            continue
        ax.set_title(f"Depth = {depth*1e9:.0f} nm")
        line.set_data(x, y)
        # ax.plot(Y[frame, :N], Y[frame, N:2*N])
        # ax.set_title(f"Depth = {depth*1e9:.0f} nm")
        # ax.set_xlim(Y[0, 0], Y[0, N-1])
        # ax.set_ylim(Y[:, N:2*N].min(), Y[:, N:2*N].max())
    return data_axes[0][1], data_axes[1][1], time_text

init()
anim = FuncAnimation(fig, update, frames=frames, interval=0.1, blit=False)

In [ ]:
from numpy.typing import NDArray
def run_simulation(Y0 : NDArray, rk4_props : rhs.CRK4Options, sim_props : rhs.CSimulationProperties, optomechanical_props : rhs.COptomechanicalProperties, adiminsionalize_time = True):
    # t1 t0 to non diamensional time
    L0 = sim_props.L / (2.0 * np.pi)
    g = 3*2.6e-24 / sim_props.depth**4
    _t0 = np.sqrt(L0 / g)

    if adiminsionalize_time:
        rk4_props.t0 =  1e-6 * rk4_props.t0 / _t0 # t0, t1 timeStep are in us, convert to non-dimensional time
        rk4_props.t1 = 1e-6 * rk4_props.t1 / _t0
        rk4_props.timeStep = 1e-6 * rk4_props.timeStep / _t0
    
    res, T_new, Y_new = simManager.integrate_augmented_optomechanical_problem(Y0, sim_props, optomechanical_props, rk4_props)
    if res != 0:
        print("Error in integration")
        exit(1)

    return T_new, Y_new, L0, _t0
# rk4_props.t1 = times[0][-1]
target_time = 5000e-6
target_timeStep = 0.01 # us


for i, depth in enumerate(depths):
    sim_props.depth = depth
    t1 = times[i][-1]
    t2 = target_time / t0s[i]

    rk4_props.t0 = t1
    rk4_props.t1 = t2
    rk4_props.timeStep = target_timeStep * 1e-6 / t0s[i]

    print(f"Integrating for depth={depth*1e9:.0f} nm with time step {rk4_props.timeStep:.2e} and total time {rk4_props.t1:.2e} a.u.t")
    T_new, Y_new, L0, _t0 = run_simulation(interfaces[i][-1, :] , rk4_props, sim_props, optomechanical_props, adiminsionalize_time=False)

    ### append the new data to the old data
    times[i] = np.concatenate((times[i], T_new))
    interfaces[i] = np.concatenate((interfaces[i], Y_new), axis=0)



    

Integrating for depth=15 nm with time step 5.45e-03 and total time 1.48e+03 a.u.t
